In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
allscores = pd.read_csv(our_data + "cache/allscores.csv")

In [3]:
allscores

,Unnamed: 0,complex_name,wei_score,if_right,enzyme,substrate,cc,substrate_cc,scores,scores0,...,scores96,rank96,scores97,rank97,scores98,rank98,scores99,rank99,scores100,rank100
0,2669,CYP72A66v2_ABA_C14,0.530,1,CYP72A66v2,ABA,C14,ABAC14,0.297941,0.297941,...,0.520718,1.0,0.523038,1.0,0.525359,1.0,0.527679,1.0,0.530,1.0
1,2670,CYP72A7_ABA_C14,0.513,1,CYP72A7,ABA,C14,ABAC14,0.103160,0.103160,...,0.496606,2.0,0.500705,2.0,0.504803,2.0,0.508902,2.0,0.513,2.0
2,2671,CYP72A62v2_ABA_C14,0.487,1,CYP72A62v2,ABA,C14,ABAC14,0.223932,0.223932,...,0.476477,4.0,0.479108,4.0,0.481739,4.0,0.484369,4.0,0.487,4.0
3,2673,CYP707A1_ABA_C14,0.418,1,CYP707A1,ABA,C14,ABAC14,0.015169,0.015169,...,0.401887,24.0,0.405915,24.0,0.409943,24.0,0.413972,23.0,0.418,21.0
4,2674,CYP72A475_ABA_C14,0.451,1,CYP72A475,ABA,C14,ABAC14,0.101858,0.101858,...,0.437034,8.0,0.440526,8.0,0.444017,7.0,0.447509,7.0,0.451,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47357,75056,CYP72A610_AHT_C8,0.300,1,CYP72A610,AHT,C8,AHTC8,0.056422,0.056422,...,0.290257,6.0,0.292693,6.0,0.295128,7.0,0.297564,7.0,0.300,8.0
47358,75057,CYP74L1_AHT_C8,0.249,1,CYP74L1,AHT,C8,AHTC8,0.040597,0.040597,...,0.240664,10.0,0.242748,10.0,0.244832,10.0,0.246916,10.0,0.249,9.0
47359,75058,CYP79A12_AHT_C8,0.248,1,CYP79A12,AHT,C8,AHTC8,0.158764,0.158764,...,0.244431,9.0,0.245323,9.0,0.246215,9.0,0.247108,9.0,0.248,10.0
47360,75059,CYP71D12_AHT_C8,0.229,1,CYP71D12,AHT,C8,AHTC8,0.045258,0.045258,...,0.221650,12.0,0.223488,12.0,0.225325,12.0,0.227163,12.0,0.229,12.0


In [ ]:
allscores76 = allscores[
    [
        "complex_name",
        "scores",
        "wei_score",
        "if_right",
        "enzyme",
        "substrate",
        "cc",
        "substrate_cc",
        "scores76",
        "rank76",
    ]
]

In [ ]:
allscores76.to_csv(our_data + "paper/allscores76.csv")

In [ ]:
combdata = pd.read_pickle(our_data + "combdata.pkl")

In [31]:
combdata

,complex_name,wei_score,if_right,enzyme,substrate,cc,substrate_cc,scores
2669,CYP72A66v2_ABA_C14,0.530,1,CYP72A66v2,ABA,C14,ABAC14,0.297941
2670,CYP72A7_ABA_C14,0.513,1,CYP72A7,ABA,C14,ABAC14,0.103160
2671,CYP72A62v2_ABA_C14,0.487,1,CYP72A62v2,ABA,C14,ABAC14,0.223932
2673,CYP707A1_ABA_C14,0.418,1,CYP707A1,ABA,C14,ABAC14,0.015169
2674,CYP72A475_ABA_C14,0.451,1,CYP72A475,ABA,C14,ABAC14,0.101858
...,...,...,...,...,...,...,...,...
75056,CYP72A610_AHT_C8,0.300,1,CYP72A610,AHT,C8,AHTC8,0.056422
75057,CYP74L1_AHT_C8,0.249,1,CYP74L1,AHT,C8,AHTC8,0.040597
75058,CYP79A12_AHT_C8,0.248,1,CYP79A12,AHT,C8,AHTC8,0.158764
75059,CYP71D12_AHT_C8,0.229,1,CYP71D12,AHT,C8,AHTC8,0.045258


In [ ]:
combdata.groupby(["enzyme", "substrate"]).size().unique()

array([ 1,  8,  2,  3,  5,  4,  9,  7,  6, 10, 11])

In [ ]:
combdata = combdata.set_index(["enzyme", "substrate"])

In [ ]:
pivoted = combdata.groupby(
    [combdata.index.get_level_values(0), combdata.index.get_level_values(1)]
).apply(
    pd.DataFrame.pivot_table,
    index=["cc"],
    columns=[],
    values="wei_score",
    aggfunc="first",
)


pivoted.reset_index(level=[0, 1], inplace=True)

In [34]:
pivoted

,enzyme,substrate,wei_score
cc,,,
C14,CYP51G1,ABA,0.329
C17,CYP51G1,ABI,0.348
C5,CYP51G1,ABS,0.269
C12,CYP51G1,ABT,0.221
C14,CYP51G1,ADI,0.326
...,...,...,...
C9,CYP99A2,TSP,0.168
C18,CYP99A2,TYP,0.403
C26,CYP99A2,UAC,0.293


In [ ]:
combdata["rank_e"] = combdata.groupby("substrate")["scores"].rank(
    method="first", ascending=False
)
combdata["rank_e_c"] = combdata.groupby("substrate_cc")["scores"].rank(
    method="first", ascending=False
)
combdata["rank_g"] = combdata.groupby("substrate")["wei_score"].rank(
    method="first", ascending=False
)
combdata["rank_g_c"] = combdata.groupby("substrate_cc")["wei_score"].rank(
    method="first", ascending=False
)

In [ ]:
combdata.to_csv(our_data + "paper/combdata.csv")